[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafzal/frontier/blob/main/examples/exact_solver_comparison/exact_solver_comparison.ipynb)

# Exact solvers as Frontier backends — coverage · optimality · speed

Frontier is a **decision layer around an exact solver**. The solver optimizes *one* problem exactly; Frontier
wraps it in an evolutionary explorer (NSGA-II) that turns that single optimal point into a whole **trustworthy
frontier** of tradeoffs — complementary, net-new value on top of the solve. The exact backend is **pluggable**:
drop any exact solver in as the per-scalarization inner solve and every returned point is optimal. We show that
with two — **HiGHS (CPU)** and **NVIDIA cuOpt (GPU)** — and where each one earns its place.

**What's inside**
1. **The wash breaks at scale.** Coverage *and* optimality of NSGA-alone vs NSGA+exact across problem size. At
   small n they tie; at scale the heuristic covers less *and* returns sub-optimal points as if they were optimal,
   while the exact solver holds 100% optimality. **This is the core win — and it's the same with either backend.**
2. **Solver-agnostic.** cuOpt (GPU) and HiGHS (CPU) produce the same frontier — the value is the *pattern*, not the vendor.
3. **Two backends, two distinct benefits.** **HiGHS (CPU)** is the accessible default — `pip install`, no GPU, and
   efficient for the *many small* solves the EA loop issues. **cuOpt (GPU)** is built for *big-and-few* — its slice
   is **scale**: one large solve where the CPU runs out of time. We show each backend where it shines.
4. **Where the GPU reaches — tested, not assumed.** Two honest open questions about cuOpt's edge, answered from the
   run: (5) a **dense mean-variance QP at scale** once Frontier adopts cuOpt's **matrix `data_model` API** (the
   term-by-term build, not cuOpt, was the O(n²) ceiling); and (6) **parallelising the independent scalarizations**
   so the GPU overlaps them — the honest flip of "the CPU wins the loop." We *measure* both; we don't assert a win.

Synthetic data; every exact point **optimal to a 0.1% gap** (not "certified"). Pick a **GPU runtime**
(*Runtime → Change runtime type → GPU*) and *Run all*. The solver-agnostic cell runs the cuOpt EA loop
(~10–15 min); the dense-QP and parallel panels add a few minutes; the arc, loop-speed, and scale panels are quick.

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print(out.stdout.strip() or "no nvidia-smi output")
except FileNotFoundError:
    print("No NVIDIA GPU detected - cuOpt cannot run. In Colab: Runtime -> Change runtime type -> GPU (T4).")

In [ ]:
# Bootstrap: clone Frontier (public) + install cuOpt (GPU) + HiGHS (CPU) + engine deps. First run ~2-3 min.
import os, subprocess, sys
REPO = "/content/frontier"
if not os.path.isdir(REPO):
    rc = subprocess.run(["git", "clone", "-q", "https://github.com/cafzal/frontier.git", REPO],
                        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode
    if rc != 0 or not os.path.isdir(REPO):
        raise RuntimeError("Could not clone github.com/cafzal/frontier - check the connection, then re-run.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--extra-index-url", "https://pypi.nvidia.com", "cuopt-cu12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "highspy",
                "pymoo>=0.6", "pydantic>=2.0", "scipy>=1.11", "pandas>=2.0", "matplotlib>=3.7"], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("Frontier on path:", REPO in sys.path)

In [ ]:
import json, os, time
import numpy as np
import matplotlib.pyplot as plt
from pymoo.indicators.hv import HV
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from engine.models import Problem
from engine.optimizer import optimize
from solvers import cuopt_backend as cb     # _build_milp_data (re-export), _solve_milp_cuopt (GPU)
from solvers import highs_backend as hb     # _solve_milp_highs (CPU exact)
# Confirms the GPU solver is live (fails if not on a cuOpt runtime):
from cuopt.linear_programming.problem import Problem as CuProblem
print("cuOpt + HiGHS import OK - both exact backends live.")

In [ ]:
_MIX = {"Growth": 30, "Digital": 20, "R&D": 20, "Maintenance": 20, "Compliance": 15, "Efficiency": 15}
_TILT = {"Growth": 1.25, "Digital": 1.15, "R&D": 1.4, "Maintenance": 0.5, "Compliance": 0.2, "Efficiency": 0.9}
_RISK = {"Growth": 55, "Digital": 45, "R&D": 78, "Maintenance": 22, "Compliance": 18, "Efficiency": 35}
_FIT = {"Growth": 70, "Digital": 72, "R&D": 60, "Maintenance": 45, "Compliance": 85, "Efficiency": 58}

def gen_capital(n, seed=11):
    """A feasible binary capital-select MILP with ~n projects (NPV/Cost/Risk/StrategicFit) under a
    budget + per-category caps + dependencies + exclusions + cardinality. Scales the committed
    120-project instance proportionally; deterministic."""
    rng = np.random.default_rng(seed)
    counts = {c: max(1, round(n * k / 120)) for c, k in _MIX.items()}
    cats = [c for c, k in counts.items() for _ in range(k)]
    n = len(cats); ids = [f"P{i+1:05d}" for i in range(n)]
    options, scores, cost_arr = [], [], []
    for pid, cat in zip(ids, cats):
        cost = float(np.round(rng.uniform(4, 45), 1))
        npv = float(np.round(max(-8, cost * _TILT[cat] * rng.uniform(0.6, 1.7) - rng.uniform(0, 10)), 1))
        risk = float(np.round(np.clip(_RISK[cat] + rng.normal(0, 12), 5, 98), 1))
        fit = float(np.round(np.clip(_FIT[cat] + rng.normal(0, 14), 10, 99), 1))
        cost_arr.append(cost); options.append({"name": pid, "description": f"{cat} initiative"})
        scores += [{"option": pid, "objective": "NPV", "value": npv}, {"option": pid, "objective": "Cost", "value": cost},
                   {"option": pid, "objective": "Risk", "value": risk}, {"option": pid, "objective": "StrategicFit", "value": fit}]
    by_cat = {c: [ids[i] for i in range(n) if cats[i] == c] for c in counts}
    budget = float(np.round(0.22 * sum(cost_arr), -1))
    forced = by_cat["Compliance"][:max(1, round(3 * n / 120))]; enablers = by_cat["Efficiency"]
    dep_src = by_cat["Growth"][:round(6 * n / 120)] + by_cat["Digital"][:round(4 * n / 120)]
    deps = [{"type": "dependency", "if_option": s, "then_option": enablers[i % len(enablers)]} for i, s in enumerate(dep_src)]
    gp, rp = by_cat["Growth"], by_cat["R&D"]; npair = min(len(gp)//2, len(rp)//2, max(1, round(5*n/120)))
    excl = ([{"type": "exclusion_pair", "option_a": gp[2*i], "option_b": gp[2*i+1]} for i in range(npair)]
            + [{"type": "exclusion_pair", "option_a": rp[2*i], "option_b": rp[2*i+1]} for i in range(npair)])
    groups = [{"type": "group_limit", "options": by_cat[c], "max": max(1, round(m*n/120))}
              for c, m in {"Growth": 8, "Digital": 6, "R&D": 6, "Maintenance": 7}.items()]
    card_lo, card_hi = max(2, round(0.15*n)), max(3, round(0.33*n))
    constraints = ([{"type": "objective_bound", "objective": "Cost", "operator": "max", "value": budget},
                    {"type": "cardinality", "min": card_lo, "max": card_hi}]
                   + [{"type": "force_include", "option": f} for f in forced] + deps + excl + groups)
    return Problem(name=f"Capital Project Selection ({n})", domain="corporate finance",
                   context=f"Choose which of {n} capital projects to fund under a ${budget:.0f}M budget.", approach="binary",
                   objectives=[{"name": "NPV", "direction": "maximize", "unit": "$M", "aggregation": "sum"},
                               {"name": "Cost", "direction": "minimize", "unit": "$M", "aggregation": "sum"},
                               {"name": "Risk", "direction": "minimize", "unit": "score", "aggregation": "sum"},
                               {"name": "StrategicFit", "direction": "maximize", "unit": "score", "aggregation": "sum"}],
                   constraints=constraints, options=options, scores=scores)

def gen_portfolio(n, n_factors=8, seed=7):
    """A DENSE mean-variance portfolio: n assets, expected returns mu, and a dense PSD covariance
    from a k-factor model (B Bᵀ + idiosyncratic diagonal). Every pair is correlated through the
    shared factors, so Σ is genuinely dense — the GPU-BLAS regime, not a sparse network — which is
    what makes the matrix data_model QP the right tool. PSD by construction; deterministic."""
    rng = np.random.default_rng(seed)
    B = rng.standard_normal((n, n_factors)) * rng.uniform(0.04, 0.22, (1, n_factors))
    cov = B @ B.T + np.diag(rng.uniform(0.01, 0.09, n))   # dense, PSD by construction
    mu = rng.uniform(0.02, 0.18, n)
    return mu, cov

def _pts(run, objs):
    return np.array([[s.objective_values[o.name] for o in objs] for s in run.solutions])

def assemble_exact(exact_sets, dirs):
    """Exact-frontier reference = nondominated union of all EXACT-backed point sets (min-space)."""
    allpts = np.vstack([np.asarray(p, float) * dirs for p in exact_sets if len(p)])
    keep = NonDominatedSorting().do(allpts, only_non_dominated_front=True)
    return allpts[keep]

def coverage_share(pts_min, exact_ref_min, lo, span, ref):
    """HV(config) / HV(exact frontier) — breadth. pts_min/exact_ref_min are in minimize space."""
    if not len(pts_min):
        return 0.0
    return float(HV(ref_point=ref)((pts_min - lo) / span) / HV(ref_point=ref)((exact_ref_min - lo) / span))

def optimality_pct(pts_min, exact_ref_min, eps=1e-9):
    """% of a config's points NOT dominated by the exact frontier — per-point exactness (depth)."""
    if not len(pts_min):
        return 0.0
    return sum(not np.any(np.all(exact_ref_min <= p + eps, axis=1) & np.any(exact_ref_min < p - eps, axis=1))
               for p in pts_min) / len(pts_min)

In [ ]:
def _sweep(Snorm, S, dirs, mc, nb, objs, inner, n_weights=300):
    """Weighted-sum exact sweep over an inner solver -> distinct frontier points (the ceiling)."""
    rng = np.random.default_rng(0)
    W = list(np.eye(len(objs))) + [w / w.sum() for w in rng.random((n_weights, len(objs)))]
    seen, pts = set(), []
    for w in W:
        sel, ok = inner(Snorm @ (np.asarray(w) * dirs), [], mc, nb)
        if not ok:
            continue
        key = tuple(round(float(S[:, j] @ sel), 3) for j in range(len(objs)))
        if key not in seen:
            seen.add(key); pts.append(list(key))
    return np.array(pts)

def _box(point_sets, exact_ref, dirs, objs):
    allmin = np.vstack([np.asarray(p, float) * dirs for p in point_sets if len(p)] + [exact_ref])
    lo, hi = allmin.min(0), allmin.max(0)
    return lo, np.where(hi > lo, hi - lo, 1.0), np.full(len(objs), 1.1)

def run_arc(scales, seed=42):
    """Coverage + optimality of NSGA-alone vs NSGA+exact (HiGHS) across problem size -> the
    wash->break arc. HiGHS stands in for the exact inner solve (solver-agnostic, and fast)."""
    rows = []
    for n in scales:
        problem = gen_capital(n); objs = problem.objectives
        nb, names, S, dirs, mc = cb._build_milp_data(problem)
        cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)
        nsga = _pts(optimize(problem, seed=seed), objs)                       # solver=None -> NSGA heuristic
        pair = _pts(optimize(problem, seed=seed, solver="highs"), objs)       # NSGA + exact inner solve (CPU)
        sweep = _sweep(Snorm, S, dirs, mc, nb, objs, hb._solve_milp_highs)
        ref = assemble_exact([sweep, pair], dirs)
        lo, span, rp = _box([nsga, pair, sweep], ref, dirs, objs)
        row = {"n": nb,
               "nsga_cov": coverage_share(nsga * dirs, ref, lo, span, rp), "nsga_opt": optimality_pct(nsga * dirs, ref),
               "pair_cov": coverage_share(pair * dirs, ref, lo, span, rp), "pair_opt": optimality_pct(pair * dirs, ref)}
        rows.append(row)
        print(f"  n={nb:>4}   NSGA-alone: cov {row['nsga_cov']:.0%} / opt {row['nsga_opt']:.0%}"
              f"      NSGA+exact: cov {row['pair_cov']:.0%} / opt {row['pair_opt']:.0%}")
    return rows

## 1 — the wash breaks at scale

NSGA-alone (heuristic) vs NSGA + an exact inner solve, across problem size, on **two** axes: **coverage** (how
much of the exact frontier you span) and **optimality** (what % of the points you return are *actually*
Pareto-optimal). At small n they sit together — a good evolutionary optimizer already covers a small frontier
and returns near-optimal points, so exact-in-the-loop washes. As the problem grows the heuristic degrades on
both axes; the pairing holds **100% optimality** throughout and pulls ahead on coverage. (HiGHS runs the exact
solve here — fast, and the next cell shows cuOpt is identical.)

In [ ]:
rows = run_arc((30, 60, 90, 120))
ns = [r["n"] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(ns, [r["nsga_cov"] for r in rows], "o-", color="darkorange", label="NSGA-alone")
ax[0].plot(ns, [r["pair_cov"] for r in rows], "s-", color="seagreen", label="NSGA + exact")
ax[1].plot(ns, [r["nsga_opt"] for r in rows], "o-", color="darkorange", label="NSGA-alone")
ax[1].plot(ns, [r["pair_opt"] for r in rows], "s-", color="seagreen", label="NSGA + exact")
ax[0].set_title("Coverage vs scale"); ax[0].set_ylabel("HV share of the exact frontier")
ax[1].set_title("Optimality vs scale (% of points actually Pareto-optimal)")
for a in ax:
    a.set_xlabel("problem size n (projects)"); a.set_ylim(0, 1.05); a.grid(alpha=0.3); a.legend(loc="lower left")
r = rows[-1]
ax[1].annotate(f"{1 - r['nsga_opt']:.0%} of NSGA's points\nare actually sub-optimal",
               xy=(r["n"], r["nsga_opt"]), xytext=(0.4, 0.30), textcoords="axes fraction", fontsize=9,
               arrowprops=dict(arrowstyle="->", color="darkorange"))
plt.tight_layout(); plt.show()

## 2 — the layer in one picture: one plan → the whole frontier (either backend)

The solver alone returns **one** optimal plan (the star). Frontier — the layer — turns that same solve into the
**whole frontier**, and the *same* frontier whether the inner solver is HiGHS (CPU) or cuOpt (GPU): same coverage,
same 100% optimality. The value is the **pattern + the layer**, not the backend. *(Slow cell: the cuOpt EA loop
issues ~750 GPU solves, ~10–15 min.)*

In [ ]:
n = 120; problem = gen_capital(n); objs = problem.objectives
nb, names, S, dirs, mc = cb._build_milp_data(problem)
cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)
pair_h = _pts(optimize(problem, seed=42, solver="highs"), objs)   # exact inner solve on CPU
pair_c = _pts(optimize(problem, seed=42, solver="cuopt"), objs)   # exact inner solve on GPU
sweep = _sweep(Snorm, S, dirs, mc, nb, objs, hb._solve_milp_highs)
solo_sel, _ = hb._solve_milp_highs(dirs[0] * S[:, 0], [], mc, nb)   # the solver alone: one optimal plan
solo = [float(S[:, j] @ solo_sel) for j in range(len(objs))]
ref = assemble_exact([sweep, pair_h, pair_c], dirs)
lo, span, rp = _box([pair_h, pair_c, sweep], ref, dirs, objs)
print(f"  solver alone:        1 plan")
for lab, P in [("NSGA + HiGHS (CPU)", pair_h), ("NSGA + cuOpt (GPU)", pair_c)]:
    print(f"  {lab}:  {len(P)} plans   coverage {coverage_share(P * dirs, ref, lo, span, rp):.0%}   optimality {optimality_pct(P * dirs, ref):.0%}")
fig, ax = plt.subplots(figsize=(7.2, 5.4))
ax.scatter([solo[0]], [solo[1]], s=240, marker="*", color="crimson", zorder=6, label="solver alone (1 plan)")
ax.scatter(pair_h[:, 0], pair_h[:, 1], s=64, color="seagreen", alpha=0.6, label=f"NSGA + HiGHS ({len(pair_h)})")
ax.scatter(pair_c[:, 0], pair_c[:, 1], s=26, color="royalblue", alpha=0.95, label=f"NSGA + cuOpt ({len(pair_c)})")
ax.set_xlabel(objs[0].name); ax.set_ylabel(objs[1].name)
ax.set_title("One plan from the solver -> the whole frontier from Frontier (same either backend)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3 — the loop regime: HiGHS (CPU) is the efficient default

The pairing issues **many small** solves — the EA's inner loop runs hundreds, each at the problem's size. At
those sizes HiGHS is **milliseconds** per solve while cuOpt's fixed GPU overhead costs ~a second; over the ~750
solves an EA frontier takes, that's the difference between minutes and much longer. So for the *many-and-small*
loop the CPU solver is the accessible, efficient default (`pip install highspy`, no GPU). **This isn't cuOpt
losing — it's the wrong regime for a GPU. Its regime, *big-and-few*, is the next panel.**

In [ ]:
cb._MILP_REL_GAP = 1e-4; cb._MILP_TIME_LIMIT = 30.0   # parity with the HiGHS backend (gap + cap)
scales = (30, 120, 250, 500, 1000)
H, C, NS = [], [], []
print(f"  {'n':>6}{'HiGHS':>11}{'cuOpt':>11}   (one bounded weighted-sum MILP solve)")
for n in scales:
    problem = gen_capital(n); nb, names, S, dirs, mc = cb._build_milp_data(problem)
    cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)
    coef = Snorm @ (np.ones(len(problem.objectives)) / len(problem.objectives) * dirs)
    t = time.perf_counter(); _, okh = hb._solve_milp_highs(coef, [], mc, nb); th = time.perf_counter() - t
    t = time.perf_counter(); _, okc = cb._solve_milp_cuopt(coef, [], mc, nb); tc = time.perf_counter() - t
    H.append(th * 1000); C.append(tc * 1000); NS.append(nb)
    print(f"  {nb:>6}{th*1000:>9.0f}ms{tc*1000:>9.0f}ms   {'ok' if (okh and okc) else 'CHECK status'}")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(NS, H, "o-", color="darkorange", label="HiGHS (CPU)")
ax.plot(NS, C, "s-", color="royalblue", label="cuOpt (GPU)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("problem size n (binary vars)"); ax.set_ylabel("one solve (ms)")
ax.set_title("Per-solve cost — cuOpt's GPU overhead vs HiGHS (the EA issues ~750 of these)")
ax.legend(); ax.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

## 4 — the scale regime: does the GPU subsolver win on one big solve?

The other regime is *big-and-few*: a single large exact solve. We give **both** backends the **same time budget**
and ask for the **best feasible plan** each can return at sizes far past the loop — n = 10k → 100k. The question
is honest and we *test* it, not assume it: **does cuOpt's GPU return a good plan where the CPU runs out of time?**
We log the incumbent objective (minimize → lower is better) plus a **build-vs-solve split**, so a slow result is
attributable — and so we can see whether cuOpt's MILP honors the time budget at all (it didn't, last run — a
candidate cuOpt issue surfaced from this exact use).

In [ ]:
import highspy   # cuOpt is imported lazily inside cuopt_budget (so this cell loads without a GPU)

def highs_budget(coef, mc, n, budget):
    """One bounded MILP solve via HiGHS within a fixed wall-clock budget.
    Returns (ok, objective, build_s, solve_s, status)."""
    t = time.perf_counter()
    h = highspy.Highs(); h.silent()
    h.setOptionValue("time_limit", float(budget)); h.setOptionValue("mip_rel_gap", 1e-4)
    x = h.addBinaries(n)
    if mc["card"] is not None:
        lo, hi = mc["card"]; h.addConstr(x.sum() >= lo); h.addConstr(x.sum() <= hi)
    for c, op, v in mc["bounds"]:
        e = (x * [float(z) for z in c]).sum(); h.addConstr(e <= v if op == "max" else e >= v)
    for i in mc["force_in"]: h.addConstr(x[i] >= 1)
    for i in mc["force_out"]: h.addConstr(x[i] <= 0)
    for a, b in mc["deps"]: h.addConstr(x[a] - x[b] <= 0)
    for a, b in mc["excl"]: h.addConstr(x[a] + x[b] <= 1)
    for grp, gmax in mc["groups"]:
        g = np.zeros(n); g[list(grp)] = 1.0; h.addConstr((x * list(g)).sum() <= gmax)
    t_build = time.perf_counter() - t
    t = time.perf_counter(); h.minimize((x * [float(c) for c in coef]).sum()); t_solve = time.perf_counter() - t
    st = h.modelStatusToString(h.getModelStatus())
    ok = (st == "Optimal") or (h.solutionStatusToString(h.getInfo().primal_solution_status) == "Feasible")
    return ok, (h.getObjectiveValue() if ok else None), t_build, t_solve, st

def cuopt_budget(coef, mc, n, budget):
    """One bounded MILP solve via cuOpt (GPU) within a fixed wall-clock budget — same model, same
    gap. Build (Python addVariable/addConstraint) is timed apart from the GPU solve."""
    import gc
    from cuopt.linear_programming import SolverSettings
    from cuopt.linear_programming.problem import INTEGER, MINIMIZE, Problem as CuP
    from cuopt.linear_programming.solver.solver_parameters import CUOPT_MIP_RELATIVE_GAP, CUOPT_TIME_LIMIT
    t = time.perf_counter()
    p = CuP("scale")
    x = [p.addVariable(lb=0.0, ub=1.0, vtype=INTEGER, name=f"x{i}") for i in range(n)]
    p.setObjective(sum(float(coef[i]) * x[i] for i in range(n)), sense=MINIMIZE)
    if mc["card"] is not None:
        lo, hi = mc["card"]; p.addConstraint(sum(x) >= lo); p.addConstraint(sum(x) <= hi)
    for c, op, v in mc["bounds"]:
        e = sum(float(c[i]) * x[i] for i in range(n)); p.addConstraint(e <= v if op == "max" else e >= v)
    for i in mc["force_in"]: p.addConstraint(x[i] >= 1)
    for i in mc["force_out"]: p.addConstraint(x[i] <= 0)
    for a, b in mc["deps"]: p.addConstraint(x[a] - x[b] <= 0)
    for a, b in mc["excl"]: p.addConstraint(x[a] + x[b] <= 1)
    for grp, gmax in mc["groups"]: p.addConstraint(sum(x[i] for i in grp) <= gmax)
    t_build = time.perf_counter() - t
    s = SolverSettings(); s.set_parameter(CUOPT_TIME_LIMIT, float(budget)); s.set_parameter(CUOPT_MIP_RELATIVE_GAP, 1e-4)
    t = time.perf_counter(); p.solve(s); t_solve = time.perf_counter() - t
    st = getattr(p.Status, "name", "?"); ok = st in ("Optimal", "FeasibleFound")
    obj = float(p.ObjValue) if ok else None
    del p; gc.collect()
    return ok, obj, t_build, t_solve, st

In [ ]:
BUDGET = 30.0   # same wall-clock budget for both backends
print(f"best feasible objective in {BUDGET:.0f}s (minimize - lower is better); b/s = build/solve seconds")
print(f"  {'n':>7} | {'HiGHS obj':>11} {'b/s':>9} {'status':>9} | {'cuOpt obj':>11} {'b/s':>9} {'status':>14}")
for n in (10000, 25000, 50000, 100000):
    problem = gen_capital(n); nb, names, S, dirs, mc = cb._build_milp_data(problem)
    cmin, cmax = S.min(0), S.max(0); Snorm = (S - cmin) / np.where(cmax > cmin, cmax - cmin, 1.0)
    coef = Snorm @ (np.ones(len(problem.objectives)) / len(problem.objectives) * dirs)
    okh, oh, bh, sh, sth = highs_budget(coef, mc, nb, BUDGET)
    okc, oc, bc, sc, stc = cuopt_budget(coef, mc, nb, BUDGET)
    print(f"  {nb:>7} | {(f'{oh:.3f}' if okh else 'NONE'):>11} {f'{bh:.0f}/{sh:.0f}':>9} {sth:>9} | "
          f"{(f'{oc:.3f}' if okc else 'NONE'):>11} {f'{bc:.0f}/{sc:.0f}':>9} {stc:>14}")
print("\nThe GPU subsolver wins at scale where HiGHS returns NONE but cuOpt returns a feasible plan.")

## 5 — the scale regime, part II: a dense QP, and the matrix API that unlocks it

Frontier's first cuOpt QP backend built the objective **term by term** through the high-level `Problem` API —
one Python `+` per covariance entry, an **O(n²)** construction that buries the GPU solve under build time on a
dense covariance. That was *our* API choice, not a cuOpt limit: cuOpt's low-level `data_model` takes the
objective and constraints **directly as CSR arrays**, so the build is a single vectorised conversion. We adopt
that path ([`_solve_qp_cuopt_matrix`](../../solvers/cuopt_backend.py)) and ask the honest scale question: with
the build ceiling gone, does the GPU solve a **large dense mean-variance QP** faster than the CPU? **Left** is
the build fix (term-by-term vs matrix); **right** is the solve itself — cuOpt GPU BLAS vs HiGHS CPU as the
dense covariance grows. We first confirm the matrix path returns the **same optimum** as the verified
term-by-term path *and* as HiGHS. *(A Theme-B shot — reported from the run, not assumed; the right panel
decides it.)*

In [ ]:
def termwise_build_secs(cov, mu, target, max_weight=None):
    """Time JUST the term-by-term cuOpt Problem build (no solve): the O(n²) Python-expression construction
    _solve_qp_cuopt does before solving — one '+' per covariance entry. Isolates the build cost the matrix
    data_model API removes. Mirrors solvers.cuopt_backend._solve_qp_cuopt's build, verbatim."""
    from cuopt.linear_programming.problem import Problem as CuProblem, MINIMIZE
    import gc
    n = len(mu); ub = float(max_weight) if max_weight else 1.0
    t = time.perf_counter()
    prob = CuProblem("build_only")
    w = [prob.addVariable(lb=0.0, ub=ub, name=f"w_{i}") for i in range(n)]
    quad = None
    for i in range(n):
        for j in range(n):
            c = float(cov[i, j])
            if abs(c) > 1e-12:
                term = c * w[i] * w[j]
                quad = term if quad is None else quad + term
    prob.setObjective(quad, sense=MINIMIZE)
    prob.addConstraint(sum(w) == 1, name="fully_invested")
    rexp = sum(float(mu[i]) * w[i] for i in range(n))
    prob.addConstraint(rexp >= float(target), name="return_target")
    dt = time.perf_counter() - t
    del prob; gc.collect()
    return dt

def matrix_build_secs(cov, mu, target, max_weight=None):
    """Time the matrix (CSR data_model) build — one vectorised numpy->scipy conversion (cb._qp_to_csr),
    exactly what _solve_qp_cuopt_matrix does before the GPU solve. O(nnz), not O(n²) Python objects."""
    t = time.perf_counter(); cb._qp_to_csr(cov, mu, target, True, max_weight); return time.perf_counter() - t

def qp_solve_secs(fn, cov, mu, target, max_weight=None):
    """Time one inner QP solve via fn (cuOpt-matrix on GPU, or HiGHS on CPU), build + solve.
    Returns (seconds, ok, weights)."""
    t = time.perf_counter(); w, ok = fn(cov, mu, target, True, max_weight, None, None)
    return time.perf_counter() - t, ok, w

In [ ]:
mu_c, cov_c = gen_portfolio(60); cov_c = cb._nearest_psd(cov_c); tgt_c = float(np.median(mu_c))
w_m, okm = cb._solve_qp_cuopt_matrix(cov_c, mu_c, tgt_c, True, None)
w_t, okt = cb._solve_qp_cuopt(cov_c, mu_c, tgt_c, True, None)
w_h, okh = hb._solve_qp_highs(cov_c, mu_c, tgt_c, True, None)
print("correctness @ n=60 (min-variance under a return floor):")
print(f"  matrix vs term-by-term  weight max-diff: {np.abs(w_m - w_t).max():.2e}   (same cuOpt math, scalable build)")
print(f"  matrix vs HiGHS         weight max-diff: {np.abs(w_m - w_h).max():.2e}   (solver-agnostic optimum)")

SCALES = (50, 100, 200, 400, 800, 1500)
TERMWISE_CAP = 400   # the O(n²) Python build is only timed where it still finishes in seconds
bt, bm, sc, sh, NS = [], [], [], [], []
print(f"\n  {'n':>6} {'build term':>12} {'build matrix':>13} {'solve cuOpt':>12} {'solve HiGHS':>12}")
for n in SCALES:
    mu, cov = gen_portfolio(n); cov = cb._nearest_psd(cov); tgt = float(np.median(mu))
    t_term = termwise_build_secs(cov, mu, tgt) if n <= TERMWISE_CAP else None
    t_mat = matrix_build_secs(cov, mu, tgt)
    s_c, okc, _ = qp_solve_secs(cb._solve_qp_cuopt_matrix, cov, mu, tgt)
    s_h, okh, _ = qp_solve_secs(hb._solve_qp_highs, cov, mu, tgt)
    bt.append(t_term); bm.append(t_mat); sc.append(s_c); sh.append(s_h); NS.append(n)
    tt = f"{t_term * 1000:>10.0f}ms" if t_term is not None else f"{'(skip)':>12}"
    print(f"  {n:>6} {tt} {t_mat * 1000:>11.0f}ms {s_c * 1000:>10.0f}ms {s_h * 1000:>10.0f}ms  {'ok' if (okc and okh) else 'CHECK'}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
tw_n = [n for n, t in zip(NS, bt) if t is not None]; tw_t = [t * 1000 for t in bt if t is not None]
ax[0].plot(tw_n, tw_t, "o-", color="firebrick", label="term-by-term Problem API (O(n²) Python)")
ax[0].plot(NS, [t * 1000 for t in bm], "s-", color="seagreen", label="matrix data_model API (CSR, vectorised)")
ax[0].set_title("Build cost — the O(n²) ceiling was the high-level API, not cuOpt"); ax[0].set_ylabel("build one QP (ms)")
ax[1].plot(NS, [t * 1000 for t in sc], "s-", color="royalblue", label="cuOpt QP (GPU, matrix)")
ax[1].plot(NS, [t * 1000 for t in sh], "o-", color="darkorange", label="HiGHS QP (CPU)")
ax[1].set_title("Solve cost — dense covariance QP, GPU BLAS vs CPU"); ax[1].set_ylabel("solve one QP (ms)")
for a in ax:
    a.set_xlabel("assets n (dense covariance)"); a.set_xscale("log"); a.set_yscale("log")
    a.grid(alpha=0.3, which="both"); a.legend(loc="upper left")
plt.tight_layout(); plt.show()

## 6 — parallelising the loop: do the independent solves overlap on the GPU?

Panel 3 showed the EA's *sequential* inner loop favours the CPU. But the scalarizations are **independent** —
nothing forces them to run one at a time. cuOpt releases the GIL during each GPU `Solve`, so a plain
`concurrent.futures` thread pool ([`_parallel_solve`](../../solvers/cuopt_backend.py)) lets the GPU **overlap**
independent solves — the DIY pattern cuOpt's deprecated `BatchSolve` now points users to. The question, the
honest flip of "the CPU wins the loop," is whether that overlap turns the GPU's per-solve overhead into a
**throughput** win across a frontier's worth of scalarizations. We run the same return-target sweep at 1/4/8/16
workers on **both** backends and measure solves-per-second; the frontier is identical at every width
(concurrency changes timing, not results). *(Theme-B shot (i): measured on the GPU run, never asserted — a flat
cuOpt curve means the solves serialise and the CPU keeps the many-small loop; a rising one means the loop-flip
is real.)*

In [ ]:
n = 300; mu, cov = gen_portfolio(n); cov = cb._nearest_psd(cov)
K = 64                      # a frontier's worth of independent return-target scalarizations
targets = np.linspace(float(np.percentile(mu, 10)), float(np.percentile(mu, 90)), K)
specs = [(cov, mu, float(t), True, None, None, None) for t in targets]   # inner-QP positional args
WORKERS = (1, 4, 8, 16)

def throughput(fn, workers):
    """solves/sec for the sweep at a given thread-pool width, + the frontier it produced (so we can
    confirm concurrency preserves results, not just timing)."""
    t = time.perf_counter()
    res = cb._parallel_solve(fn, specs, max_workers=workers)
    dt = time.perf_counter() - t
    pts = sorted((round(float(np.sqrt(w @ cov @ w)), 6), round(float(mu @ w), 6)) for w, ok in res if ok)
    return len(specs) / dt, dt, pts

print(f"  {K} independent QP scalarizations, dense n={n}")
print(f"  {'workers':>8} {'cuOpt sol/s':>13} {'cuOpt s':>9} {'HiGHS sol/s':>13} {'HiGHS s':>9}  frontier")
cu_tp, hi_tp, base = [], [], None
for W in WORKERS:
    tc, dc, pc = throughput(cb._solve_qp_cuopt_matrix, W)
    th, dh, ph = throughput(hb._solve_qp_highs, W)
    cu_tp.append(tc); hi_tp.append(th)
    base = base or pc
    same = "ok" if len(pc) == len(base) else "CHECK"
    print(f"  {W:>8} {tc:>13.1f} {dc:>8.2f}s {th:>13.1f} {dh:>8.2f}s   {same}")

fig, ax = plt.subplots(figsize=(7.4, 5))
ax.plot(WORKERS, cu_tp, "s-", color="royalblue", label="cuOpt (GPU) — does it overlap?")
ax.plot(WORKERS, hi_tp, "o-", color="darkorange", label="HiGHS (CPU cores)")
ax.set_xlabel("thread-pool workers"); ax.set_ylabel("scalarizations / second")
ax.set_title(f"Parallel throughput over {K} independent solves (1 = the sequential loop)")
ax.set_xticks(WORKERS); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()
print("\nRising cuOpt curve = the GPU overlaps the independent solves (the loop-flip). Flat = they serialise "
      "and the CPU keeps the many-small loop. The run decides — we don't assert it here.")

## What this shows

- **The core win — exact + approximate, regardless of backend.** At small n NSGA-alone already covers the frontier
  and returns near-optimal points. By n=120 it covers ~60% **and** ~44% of its points are dominated — sub-optimal
  plans presented as the efficient set. NSGA + an exact inner solve stays at **100% optimality** and closes the
  coverage gap. The value isn't *more* frontier; it's a frontier you can **trust** — and it's the same with cuOpt or
  HiGHS. (On *continuous* problems the exact solver also hands you the shadow prices; on a MILP there are no duals.)
- **Two backends, two regimes — each with a real benefit.** **HiGHS (CPU)** owns the *many-small* exploration loop:
  accessible (`pip install`, no GPU) and millisecond solves. **cuOpt (GPU)** owns *big-and-few*: one large solve at a
  scale where the CPU runs out of time (Panel 4 — measured, not assumed). Pick the backend by the regime, not the logo.
- **Two GPU shots, reported honestly (Panels 5–6).** The dense-QP ceiling was *Frontier's* term-by-term API, not cuOpt:
  the matrix `data_model` build removes it (Panel 5, left), which is the single highest-value backend fix and is
  equivalent — same optimum — to the verified path. Whether the GPU then *wins* the dense solve (Panel 5, right) and
  whether parallelising the independent scalarizations flips the loop (Panel 6) are decided by the numbers each run
  prints — not claimed here. Absence of a win is reported as such; the trust result above stands either way.

**Net:** Frontier is the **decision layer** — it turns a solver's single optimal point into a frontier you can
*trust*, with any exact backend, and the GPU adds reach at scale. Complementary, not competing: the solver
optimizes; Frontier makes it a decision. As we hand more optimization to agents and heuristics, the scarce
resource isn't speed; it's **grounding**.

*Synthetic data; every exact point optimal to a 0.1% gap; duals/shadow-prices exist for continuous QP only, never a MILP.*